## Generate outputs

Given that you have already run the Ingest and Run notebooks, this notebook takes the outputs of Run (the allpixels and allfires dataframes) and genarates the archival output files.

What we want out of this notebook is:
 - a snapshot of all the fires at a given t
 - a timeseries of each fire across time

In [1]:
import os
import datetime
import pandas as pd
import geopandas as gpd

from fireatlas import FireTime, FireObj, postprocess
from fireatlas.utils import timed

region = ['Quebec_PostHoc',]  # note you don't need the shape in here, just the name
tst = [2023, 4, 30, 'AM']
ted = [2023, 9, 15, 'PM']

2024-09-25 18:49:09,682 - fireatlas.FireLog - INFO - logger initialized!


## Read from disk

Since we want to use precisely the files that we just created in the Run notebook. We will set the `location` to "local".

In [2]:
allpixels = postprocess.read_allpixels(tst, ted, region, location="local")

2024-09-25 18:49:15,939 - fireatlas.FireLog - INFO - func:read_allpixels took: 1.75 sec


In [3]:
allfires_gdf = postprocess.read_allfires_gdf(tst, ted, region, location="local")

2024-09-25 18:49:16,444 - fireatlas.FireLog - INFO - func:read_allfires_gdf took: 500.29 ms


## Write snapshots

Write each geometry object into its own flatgeobuf file within a subdirectory.

In [4]:
%%time
postprocess.save_snapshots(allfires_gdf, region, tst, ted)

2024-09-25 18:51:18,831 - fireatlas.FireLog - INFO - func:save_snapshots took: 2.01 min


CPU times: user 50.9 s, sys: 2.43 s, total: 53.4 s
Wall time: 2min


[]

## Write large fires

Start by getting a list containing all the fireIDs for the large fires in the allfires geodataframe.

In [5]:
large_fires = postprocess.find_largefires(allfires_gdf)

2024-09-25 18:51:18,853 - fireatlas.FireLog - INFO - func:find_largefires took: 9.27 ms


First we'll use the `allpixels` object to create the `nplist` layer

In [6]:
postprocess.save_large_fires_nplist(allpixels, region, large_fires, tst)

2024-09-25 18:51:37,791 - fireatlas.FireLog - INFO - func:save_large_fires_nplist took: 18.93 sec


The rest of the layers will be created directly from the `allfires_gdf`.

In [7]:
postprocess.save_large_fires_layers(allfires_gdf, region, large_fires, tst, ted)

2024-09-25 18:51:55,026 - fireatlas.FireLog - INFO - func:fill_activefire_rows took: 17.22 sec


20325 rows that potentially need a merge


2024-09-25 18:52:50,357 - fireatlas.FireLog - INFO - func:save_combined_large_fire_layers took: 5.87 sec
2024-09-25 18:52:50,439 - fireatlas.FireLog - INFO - func:save_large_fires_layers took: 1.21 min
